In [ ]:
import re
import glob
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.gridspec import GridSpec
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'font.family':      'DejaVu Sans',
    'font.size':        11,
    'axes.titlesize':   12,
    'axes.labelsize':   11,
    'xtick.labelsize':  10,
    'ytick.labelsize':  10,
    'legend.fontsize':  10,
    'axes.linewidth':   0.8,
    'axes.grid':        True,
    'grid.linestyle':   '--',
    'grid.linewidth':   0.5,
    'grid.alpha':       0.6,
    'figure.dpi':       150,
    'savefig.dpi':      300,
    'savefig.bbox':     'tight',
})

In [ ]:
# ── Parse .out files ──────────────────────────────────────────────────────────
# Filename convention: {ranks}_{omp_threads}.out

BENCH_DIR  = os.path.dirname(os.path.abspath('__file__'))  # same dir as notebook
OMP_FILTER = 64   # only keep runs with this OMP thread count

def parse_out_file(path):
    fname  = os.path.basename(path)
    stem   = fname.replace('.out', '')
    parts  = stem.split('_')
    ranks  = int(parts[0])
    omp    = int(parts[1])

    with open(path) as f:
        text = f.read()

    def extract(pattern):
        m = re.search(pattern, text)
        return float(m.group(1)) if m else None

    throughput = extract(r'Throughput\s*:\s*([0-9]+\.?[0-9]*)')
    latency    = extract(r'Mean latency\s*:\s*([0-9]+\.?[0-9]*)')
    std        = extract(r'Std dev\s*:\s*([0-9]+\.?[0-9]*)')

    if throughput is None or throughput == 0 or latency == 0:
        return None

    return dict(ranks=ranks, omp=omp,
                throughput=throughput, latency=latency, std=std)


pattern = os.path.join(BENCH_DIR, f'*_{OMP_FILTER}.out')
records = []
for path in sorted(glob.glob(pattern)):
    rec = parse_out_file(path)
    if rec is not None:
        records.append(rec)
        print(f"  {os.path.basename(path):20s}  ranks={rec['ranks']:3d}  "
              f"throughput={rec['throughput']:>10,.1f} s/s  "
              f"latency={rec['latency']:.2f} ms")

records.sort(key=lambda r: r['ranks'])
print(f"\nLoaded {len(records)} valid runs with OMP_NUM_THREADS={OMP_FILTER}")

In [ ]:
# ── Derived arrays ────────────────────────────────────────────────────────────
ranks      = np.array([r['ranks']      for r in records])
throughput = np.array([r['throughput'] for r in records])
latency    = np.array([r['latency']    for r in records])
std_dev    = np.array([r['std']        for r in records])

# Weak scaling efficiency: latency(N) / latency(1)
# Ideal = 1.0 (latency constant regardless of rank count)
baseline_latency    = latency[0]
baseline_throughput = throughput[0]
efficiency = baseline_latency / latency * 100  # 100% = no overhead

print(f"{'Ranks':>6} {'Throughput':>14} {'Latency':>10} {'Std':>8} {'Efficiency':>11}")
print("-" * 57)
for i in range(len(ranks)):
    print(f"{ranks[i]:>6}  {throughput[i]:>12,.0f}/s  "
          f"{latency[i]:>8.2f}ms  {std_dev[i]:>6.2f}ms  "
          f"{efficiency[i]:>10.1f}%")

In [ ]:
# ── Colours ───────────────────────────────────────────────────────────────────
C_ACTUAL = '#2166AC'
C_IDEAL  = '#D73027'
C_EFF    = '#1A9850'
marker_kw = dict(lw=2.0, ms=7, markerfacecolor='white', markeredgewidth=2)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
ax1, ax2 = axes

# ─── (a) Throughput — should scale linearly ───────────────────────────────────
ideal_tp = baseline_throughput * ranks
ax1.fill_between(ranks, throughput, ideal_tp, alpha=0.12, color=C_IDEAL)
ax1.plot(ranks, ideal_tp,   '--', color=C_IDEAL,  **marker_kw, label='Ideal (linear)')
ax1.plot(ranks, throughput, 'o-', color=C_ACTUAL, **marker_kw, label='Measured')
for r, tp in zip(ranks, throughput):
    ax1.annotate(f'{tp/1000:.0f}k', xy=(r, tp),
                 xytext=(0, 8), textcoords='offset points',
                 ha='center', fontsize=8.5, color=C_ACTUAL)
ax1.set_xscale('log', base=2)
ax1.set_xticks(ranks)
ax1.get_xaxis().set_major_formatter(ticker.ScalarFormatter())
ax1.set_xlabel('Number of MPI Ranks')
ax1.set_ylabel('Throughput (samples / sec)')
ax1.set_title('(a) Throughput (ideal: linear)')
ax1.legend(framealpha=0.85)

# ─── (b) Latency — should stay constant ──────────────────────────────────────
ax2.axhline(baseline_latency, color=C_IDEAL, ls='--', lw=1.4,
            label=f'Ideal ({baseline_latency:.1f} ms)')
ax2.errorbar(ranks, latency, yerr=std_dev, fmt='s-', color=C_ACTUAL,
             **marker_kw, capsize=4, capthick=1.5, elinewidth=1.2,
             label='Mean latency ± std')
for r, lat in zip(ranks, latency):
    ax2.annotate(f'{lat:.1f}', xy=(r, lat),
                 xytext=(0, 9), textcoords='offset points',
                 ha='center', fontsize=8.5, color=C_ACTUAL)
ax2.set_xscale('log', base=2)
ax2.set_xticks(ranks)
ax2.get_xaxis().set_major_formatter(ticker.ScalarFormatter())
ax2.set_xlabel('Number of MPI Ranks')
ax2.set_ylabel('Mean Batch Latency (ms)')
ax2.set_title('(b) Per-Batch Latency (ideal: constant)')
ax2.legend(framealpha=0.85)

fig.suptitle(
    f'MPI Weak Scaling — MLP Inference on CIFAR-10 Embeddings\n'
    f'Dardel HPC (AMD EPYC 7742, Slingshot) · '
    f'{OMP_FILTER} OMP threads/rank · 8 batches/rank · batch\_size=128',
    fontsize=12, y=1.02
)

plt.tight_layout()
plt.savefig(os.path.join(BENCH_DIR, 'mpi_weak_scaling.pdf'))
plt.savefig(os.path.join(BENCH_DIR, 'mpi_weak_scaling.png'))
plt.show()
print('Saved mpi_weak_scaling.pdf / .png')